## 使用 Responses API 提升推理模型的性能 

### 概述

将 Responses API 与 OpenAI 最新的推理模型结合使用，你可以在应用中获得更高的智能水平、更低的成本和更高效的 token 使用。该 API 还支持获取推理摘要、使用托管工具等功能，并且其设计能够适应即将推出的增强功能，带来更大的灵活性和更高的性能。



我们最近发布了两款全新的最先进推理模型 o3 和 o4-mini，它们擅长将推理能力与智能体式工具调用相结合。很多人不知道的是，充分利用我们（相对）较新的 Responses API 可以进一步提升它们的性能。本 cookbook 展示了如何充分发挥这些模型的能力，并探讨推理和函数调用在幕后是如何运作的。通过让模型访问之前的推理项（reasoning items），我们可以确保它以最高的智能水平和最低的成本运行。

我们在单独的 [cookbook](https://cookbook.openai.com/examples/responses_api/responses_example) 和 [API 参考](https://platform.openai.com/docs/api-reference/responses)中介绍了 Responses API。要点是：Responses API 与 Completions API 类似，但有所改进并增加了新功能。我们还为 Responses 推出了加密内容功能，这让它对那些无法以有状态方式使用 API 的用户更加实用！

## 推理模型的工作原理

在深入了解 Responses API 能带来哪些帮助之前，我们先快速回顾一下[推理模型](https://platform.openai.com/docs/guides/reasoning?api-mode=responses)的工作原理。o3 和 o4-mini 等模型会逐步拆解问题，生成编码其推理过程的内部思维链。出于安全考虑，这些推理 token 只会以摘要形式向用户展示。

在多步对话中，每一轮结束后推理 token 都会被丢弃，而每一步的输入和输出 token 会被传入下一步

![reasoning-context](../../images/reasoning-turns.png)
图表引自我们的[文档](https://platform.openai.com/docs/guides/reasoning?api-mode=responses#how-reasoning-works)

我们来看看返回的 response 对象：

In [3]:
from openai import OpenAI
import os
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
response = client.responses.create(
    model="o4-mini",
    input="tell me a joke",
)


In [9]:
import json

print(json.dumps(response.model_dump(), indent=2))


{
  "id": "resp_6820f382ee1c8191bc096bee70894d040ac5ba57aafcbac7",
  "created_at": 1746989954.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "o4-mini-2025-04-16",
  "object": "response",
  "output": [
    {
      "id": "rs_6820f383d7c08191846711c5df8233bc0ac5ba57aafcbac7",
      "summary": [],
      "type": "reasoning",
      "status": null
    },
    {
      "id": "msg_6820f3854688819187769ff582b170a60ac5ba57aafcbac7",
      "content": [
        {
          "annotations": [],
          "text": "Why don\u2019t scientists trust atoms?  \nBecause they make up everything!",
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "max_output_tokens": null,
  "previous_response_id": null,
  "reasoning": {
    "effort": "medium",
    "generate

从 response 对象的 JSON 输出中可以看到，除了 `output_text` 之外，模型还生成了一个推理项。该项代表模型的内部推理 token，并以 ID 的形式暴露出来——例如这里的 `rs_6820f383d7c08191846711c5df8233bc0ac5ba57aafcbac7`。由于 Responses API 是有状态的，这些推理 token 会被持久保存：只需在后续消息中包含它们的 ID，就能让之后的响应访问相同的推理项。如果你在多轮对话中使用 `previous_response_id`，模型将自动获得之前生成的所有推理项。

你还可以看到模型生成了多少推理 token。例如，在 10 个输入 token 的情况下，响应包含 148 个输出 token——其中 128 个是推理 token，不会显示在最终的助手消息中。

等等——图中不是显示之前轮次的推理会被丢弃吗？那为什么还要在后续轮次中把它传回去？

问得好！在典型的多轮对话中，你不需要包含推理项或推理 token——模型经过训练，即使没有它们也能产生最佳输出。然而，涉及工具调用时情况就不同了。如果某一轮包含函数调用（可能需要在 API 之外额外往返一次），你就需要包含推理项——可以通过 `previous_response_id`，也可以显式地将推理项添加到 `input` 中。我们通过一个简单的函数调用示例来看看它是如何工作的。

In [14]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

context = [{"role": "user", "content": "What's the weather like in Paris today?"}]

response = client.responses.create(
    model="o4-mini",
    input=context,
    tools=tools,
)


response.output

[ResponseReasoningItem(id='rs_68210c71a95c81919cc44afadb9d220400c77cc15fd2f785', summary=[], type='reasoning', status=None),
 ResponseFunctionToolCall(arguments='{"latitude":48.8566,"longitude":2.3522}', call_id='call_9ylqPOZUyFEwhxvBwgpNDqPT', name='get_weather', type='function_call', id='fc_68210c78357c8191977197499d5de6ca00c77cc15fd2f785', status='completed')]

经过一番推理后，o4-mini 模型判断它需要更多信息，于是调用一个函数来获取信息。我们可以调用该函数并将其输出返回给模型。关键在于，为了最大化模型的智能水平，我们应当包含推理项——只需将所有输出原样加回到下一轮的上下文中即可。

In [15]:
context += response.output # Add the response to the context (including the reasoning item)

tool_call = response.output[1]
args = json.loads(tool_call.arguments)


# calling the function
result = get_weather(args["latitude"], args["longitude"]) 

context.append({                               
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

# we are calling the api again with the added function call output. Note that while this is another API call, we consider this as a single turn in the conversation.
response_2 = client.responses.create(
    model="o4-mini",
    input=context,
    tools=tools,
)

print(response_2.output_text)

The current temperature in Paris is 16.3°C. If you’d like more details—like humidity, wind speed, or a brief description of the sky—just let me know!


虽然这个简单示例可能无法清楚地体现其优势——因为无论有没有推理项，模型很可能都表现良好——但我们自己的测试结果却并非如此。在 SWE-bench 等更严格的基准测试中，在相同的提示词和设置下，包含推理项带来了约 **3% 的提升**。

## 缓存

如上所示，推理模型会同时生成推理 token 和补全 token，API 对二者的处理方式不同。这种区别会影响缓存的工作方式，进而影响性能和延迟。下图展示了这些概念：

![reasoning-context](../../images/responses-diagram.png)

在第 2 轮中，来自第 1 轮的所有推理项都会被忽略并移除，因为模型不会复用之前轮次的推理项。因此，图中的第四次 API 调用无法实现完全缓存命中，因为提示词中缺少了这些推理项。不过，包含它们并无害处——API 会直接丢弃与当前轮次无关的推理项。请记住，缓存只对超过 1024 个 token 的提示词生效。在我们的测试中，从 Completions API 切换到 Responses API 后，缓存利用率从 40% 提升到了 80%。更高的缓存利用率意味着更低的成本（例如，`o4-mini` 的缓存输入 token 比未缓存的便宜 75%）以及更低的延迟。

## 加密推理项

一些组织——例如有[零数据保留（ZDR）](https://openai.com/enterprise-privacy/)要求的组织——由于合规或数据保留政策，无法以有状态的方式使用 Responses API。为支持这些场景，OpenAI 提供了[加密推理项](https://platform.openai.com/docs/guides/reasoning?api-mode=responses#encrypted-reasoning-items)，让你在保持工作流无状态的同时，仍然能从推理项中获益。

要使用加密推理项：
- 在 API 调用的 `include` 字段中添加 `["reasoning.encrypted_content"]`。
- API 将返回推理 token 的加密版本，你可以像普通推理项一样在后续请求中将其传回。

对于 ZDR 组织，OpenAI 会自动强制设置 `store=false`。当请求包含 `encrypted_content` 时，它会在内存中解密（绝不写入磁盘），用于生成下一个响应，然后被安全丢弃。任何新的推理 token 都会立即加密并返回给你，确保不会持久化任何中间状态。

下面通过一段简单的代码更新来展示其工作方式：

In [39]:
context = [{"role": "user", "content": "What's the weather like in Paris today?"}]

response = client.responses.create(
    model="o3",
    input=context,
    tools=tools,
    store=False, #store=false, just like how ZDR is enforced
    include=["reasoning.encrypted_content"] # Encrypted chain of thought is passed back in the response
)

In [34]:
# take a look at the encrypted reasoning item
print(response.output[0]) 

ResponseReasoningItem(id='rs_6821243503d481919e1b385c2a154d5103d2cbc5a14f3696', summary=[], type='reasoning', status=None, encrypted_content='gAAAAABoISQ24OyVRYbkYfukdJoqdzWT-3uiErKInHDC-lgAaXeky44N77j7aibc2elHISjAvX7OmUwMU1r7NgaiHSVWL5BtWgXVBp4BMFkWZpXpZY7ff5pdPFnW3VieuF2cSo8Ay7tJ4aThGUnXkNM5QJqk6_u5jwd-W9cTHjucw9ATGfGqD2qHrXyj6NEW9RmpWHV2SK41d5TpUYdN0xSuIUP98HBVZ2VGgD4MIocUm6Lx0xhRl9KUx19f7w4Sn7SCpKUQ0zwXze8UsQOVvv1HQxk_yDosbIg1SylEj38H-DNLil6yUFlWI4vGWcPn1bALXphTR2EwYVR52nD1rCFEORUd7prS99i18MUMSAhghIVv9OrpbjmfxJh8bSQaHu1ZDTMWcfC58H3i8KnogmI7V_h2TKAiLTgSQIkYRHnV3hz1XwaUqYAIhBvP6c5UxX-j_tpYpB_XCpD886L0XyJxCmfr9cwitipOhHr8zfLVwMI4ULu-P3ftw7QckIVzf71HFLNixrQlkdgTn-zM6aQl5BZcJgwwn3ylJ5ji4DQTS1H3AiTrFsEt4kyiBcE2d7tYA_m3G8L-e4-TuTDdJZtLaz-q8J12onFaKknGSyU6px8Ki4IPqnWIJw8SaFMJ5fSUYJO__myhp7lbbQwuOZHIQuvKutM-QUuR0cHus_HtfWtZZksqvVCVNBYViBxD2_KvKJvR-nN62zZ8sNiydIclt1yJfIMkiRErfRTzv92hQaUtdqz80UiW7FBcN2Lnzt8awXCz1pnGyWy_hNQe8C7W35zRxJDwFdb-f3VpanJT0tNmU5bfEWSXcIVmiMZL1clwzVNryf9Gk482LaWPwhVYrh

设置 `include=["reasoning.encrypted_content"]` 后，我们现在可以在返回的推理项中看到一个 `encrypted_content` 字段。该加密内容代表模型的推理状态，完全保存在客户端，OpenAI 不保留任何数据。然后，我们可以像之前处理推理项那样将其传回。

In [40]:
context += response.output # Add the response to the context (including the encrypted chain of thought)
tool_call = response.output[1]
args = json.loads(tool_call.arguments)



result = 20 #mocking the result of the function call

context.append({                               
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model="o3",
    input=context,
    tools=tools,
    store=False,
    include=["reasoning.encrypted_content"]
)

print(response_2.output_text)

It’s currently about 20 °C in Paris.


只需对 `include` 字段做一个简单的修改，我们现在就可以传回加密推理项，并利用它在智能水平、成本和延迟方面提升模型的表现。

现在，你应该已经掌握了充分利用我们最新推理模型所需的全部知识！

## 推理摘要

Responses API 的另一个实用功能是支持推理摘要。虽然我们不会公开原始的思维链 token，但用户可以访问它们的[摘要](https://platform.openai.com/docs/guides/reasoning?api-mode=responses#reasoning-summaries)。

In [9]:
# Make a hard call to o3 with reasoning summary included

response = client.responses.create(
    model="o3",
    input="What are the main differences between photosynthesis and cellular respiration?",
    reasoning={"summary": "auto"},

    
)

# Extract the first reasoning summary text from the response object
first_reasoning_item = response.output[0]  # Should be a ResponseReasoningItem
first_summary_text = first_reasoning_item.summary[0].text if first_reasoning_item.summary else None
print("First reasoning summary text:\n", first_summary_text)



First reasoning summary text:
 **Analyzing biological processes**

I think the user is looking for a clear explanation of the differences between certain processes. I should create a side-by-side comparison that lists out key elements like the formulas, energy flow, locations, reactants, products, organisms involved, electron carriers, and whether the processes are anabolic or catabolic. This structured approach will help in delivering a comprehensive answer. It’s crucial to cover all aspects to ensure the user understands the distinctions clearly.


推理摘要文本能让用户一窥模型的思考过程。例如，在包含多次函数调用的对话中，用户无需等待最终的助手消息，就能看到调用了哪些函数以及每次调用背后的推理。这为应用的用户体验增添了透明度和交互性。

## 总结

借助 OpenAI Responses API 和最新的推理模型，你可以在应用中获得更高的智能水平、更好的透明度和更高的效率。无论你是使用推理摘要、为满足合规要求使用加密推理项，还是针对成本和延迟进行优化，这些工具都能帮助你构建更稳健、更具交互性的 AI 体验。

祝你构建愉快！